# Partner-distance decoding — end-to-end demo

Decode the continuous Euclidean distance between a focal (implanted) animal and a
*specific* partner from the focal's neural activity, using regression
([`ephys/decode_partner_distance.py`](decode_partner_distance.py)).

Walks through:

1. Synthetic, ephys-aligned arrays (so the notebook runs offline).
2. Population decode — cross-validated R² / RMSE, plus the **partial R²** that
   removes the focal animal's own speed + position (the self-motion confound).
3. Single-cell scores — per-cell R², Pearson r, partial R², and the ranked
   "distance cells", plus 1-D distance tuning curves.
4. A circular-shift **null** for significance.
5. All five plots.
6. A drop-in **real-data variant** via `decode_partner_distance(session, ...)`.

The math is whole-session, time-binned: firing rates and distance are binned on
one common ephys-second grid and regressed with contiguous-block CV (no
shuffling) because both signals are strongly autocorrelated.

In [ ]:
import sys
from pathlib import Path

# Make sibling modules importable from the notebook's location (ephys/).
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

from ephys.decode_partner_distance import (
    _analyze,
    single_cell_distance_scores,
    population_distance_regression,
    compute_distance_tuning,
)
from ephys.decode_partner_distance_plots import (
    plot_distance_tuning_curves,
    plot_per_cell_r2_distribution,
    plot_predicted_vs_actual_scatter,
    plot_predicted_vs_actual_timeseries,
    plot_partner_distance_summary,
)

## 0. Synthetic, ephys-aligned arrays

These mirror exactly what `build_distance_binned_data` returns from a real
`MultiAnimalSession`:

- `firing_rates` — `(n_bins, n_cells)` Hz,
- `distance` — `(n_bins,)` focal-partner distance (here in cm),
- `nuisance` — `(n_bins, 3)` focal self-motion `[speed, x, y]`,
- `bin_centers` — `(n_bins,)` ephys seconds.

Partner distance is a mean-reverting (Ornstein-Uhlenbeck) process — bounded and
autocorrelated, like a real arena distance. We plant **two genuine distance
cells**, **one self-motion ("confound") cell** that tracks the focal's own speed
(which is itself correlated with distance), and a bank of **noise cells**.

In [ ]:
rng = np.random.default_rng(0)
T = 2000
bin_size_sec = 0.5
bin_centers = np.arange(T) * bin_size_sec

# --- Mean-reverting partner distance (OU process), in cm ----------------------
theta, mu, sigma = 0.02, 60.0, 3.0
d = np.empty(T); d[0] = mu
for t in range(1, T):
    d[t] = d[t-1] + theta * (mu - d[t-1]) + sigma * rng.standard_normal()
distance = np.abs(d)
dz = (distance - distance.mean()) / distance.std()

# --- Focal self-motion (the confound): correlated with distance ---------------
focal_speed = 0.7 * dz + 0.5 * rng.standard_normal(T)
focal_x = np.cumsum(0.3 * rng.standard_normal(T))   # arbitrary position columns
focal_y = np.cumsum(0.3 * rng.standard_normal(T))
nuisance = np.column_stack([focal_speed, focal_x, focal_y])
nuisance_names = ["focal_speed", "focal_x", "focal_y"]

# --- Cells (baseline + signal), clipped to look like Hz -----------------------
base = 5.0
c_dist_pos  =  1.0 * dz + 0.40 * rng.standard_normal(T)   # genuine distance cell (+)
c_dist_neg  = -0.8 * dz + 0.50 * rng.standard_normal(T)   # genuine distance cell (-)
c_selfmotion = focal_speed + 0.10 * rng.standard_normal(T)  # encodes self-motion only
noise_cells = rng.standard_normal((T, 12))

firing_rates = np.clip(
    base + np.column_stack([c_dist_pos, c_dist_neg, c_selfmotion, noise_cells]),
    0.0, None,
)
cell_labels = ["dist+ (0)", "dist- (1)", "self-motion (2)"] + \
              [f"noise ({i})" for i in range(3, firing_rates.shape[1])]

print(f"firing_rates : {firing_rates.shape}  (n_bins, n_cells)")
print(f"distance     : {distance.shape}  range {distance.min():.1f}-{distance.max():.1f} cm")
print(f"nuisance     : {nuisance.shape}  {nuisance_names}")

## 1. Population decode

`_analyze` is the I/O-free core shared by the CLI and GUI. It runs the
single-cell scores, the population ridge, the tuning curves, and the null in one
call and returns the full result dict.

In [ ]:
result = _analyze(
    firing_rates, distance, nuisance, bin_centers,
    units="cm", focal="631", partner="632",
    alpha=1.0, cv_folds=5,
    n_distance_bins=15, tuning_smoothing_sigma=1.0,
    null="shuffle", n_shuffles=100,
    nuisance_names=nuisance_names, seed=0,
)

print("status              :", result["status"])
print(f"population CV R2     : {result['cv_r2']:.3f}")
print(f"  partial R2 (beyond self-motion) : {result['cv_r2_partial']:.3f}")
print(f"RMSE                : {result['rmse']:.2f} {result['units']}")
print(f"null R2             : {result['null_r2']:.3f} +/- {result['null_r2_std']:.3f}")

The population CV R² sits well above the circular-shift null. The **partial
R²** is lower than the raw R² — that gap is the distance information the
population carries *beyond* the focal animal's own speed and position.

## 2. Single-cell scores + the confound

`single_cell_distance_scores` ranks cells by partial R² (when a nuisance block
is given). Watch cell 2: it has a healthy **raw** R² (it tracks self-motion,
which correlates with distance) but its **partial** R² collapses to ~0 — it
carries no distance information beyond self-motion. The two genuine distance
cells stay high in both.

In [ ]:
sc = single_cell_distance_scores(firing_rates, distance, alpha=1.0, cv_folds=5,
                                 nuisance=nuisance)
print("ranked cells (by partial R2):", sc["cell_ranking"][:5])
print()
print(f"{'cell':<18}{'raw R2':>9}{'partial R2':>12}{'pearson':>10}")
for j in list(sc["cell_ranking"][:6]) + [2]:
    print(f"{cell_labels[j]:<18}{sc['r2_per_cell'][j]:>9.3f}"
          f"{sc['r2_partial_per_cell'][j]:>12.3f}{sc['pearson_r_per_cell'][j]:>10.3f}")

## 3. Plots

All five plotting helpers take the result dict and return a Matplotlib figure.

In [ ]:
fig = plot_partner_distance_summary(result)
plt.show()

In [ ]:
fig = plot_distance_tuning_curves(result, n_top=8)
plt.show()

In [ ]:
fig = plot_per_cell_r2_distribution(result)   # raw vs partial, with null line
plt.show()

In [ ]:
fig = plot_predicted_vs_actual_scatter(result)
plt.show()
fig = plot_predicted_vs_actual_timeseries(result)
plt.show()

## 4. Real-data variant (uncomment when the SMB shares are mounted)

`decode_partner_distance` does the whole thing from a `MultiAnimalSession`:
bins the focal's rates + the focal-partner distance + self-motion nuisance on a
common ephys grid (via `get_common_binned_rates` / `get_tracking_on_ephys_clock`),
then runs the same `_analyze`. Distances come back in **cm** when the cohort
config sets `pixels_per_cm`, else in pixels (`result['units']`).

In [ ]:
# from ingestion.multi_animal_session import MultiAnimalSession
# from ephys.decode_partner_distance import decode_partner_distance
#
# session = MultiAnimalSession(
#     session_id="20251216",
#     animal_ids=["631", "632"],   # [focal (implanted), partner]
#     config_path=None,            # cohort 7 default
# )
# result = decode_partner_distance(
#     session, focal="631", partner="632",
#     bin_size=0.5, smoothing_sigma_sec=0.25,
#     alpha=1.0, cv_folds=5, null="shuffle", n_shuffles=100,
# )
# print("CV R2:", round(result["cv_r2"], 3),
#       "| partial:", round(result["cv_r2_partial"], 3),
#       "| null:", round(result["null_r2"], 3),
#       "| units:", result["units"])
# plot_partner_distance_summary(result); plt.show()
#
# # Equivalent from a shell:
# #   python -m ephys.run_partner_distance --session_id 20251216 \
# #       --animal_ids 631 632 --bin_size 0.5 --smoothing 0.25 \
# #       --null shuffle --n_shuffles 100 --output_dir ./results